# Graf-Kompass GraphRAG: Kommunale Klimaanpassung

Dieses Notebook nutzt den UBA-Bericht zur kommunalen Klimaanpassung als Einstieg in das GraphRAG-Pattern: Quelle laden, Ontologie verstehen, Dokument- und Ontologiegraph in Neo4j aufbauen, Retrieval prüfen und eine quellenbasierte Antwort erzeugen.

**Leitfrage:** Wie entsteht kommunale Klimaresilienz aus Bestandsaufnahme, Einflussfaktoren, Hebelpunkten und Maßnahmen gegen Hitze und Starkregen?

**Done ist erreicht**, wenn `outputs/klima_kommunale_klimaanpassung/provenance.json`, `outputs/klima_kommunale_klimaanpassung/provenance.html`, `outputs/klima_kommunale_klimaanpassung/retrieval_brief.md`, `outputs/klima_kommunale_klimaanpassung/answer_context.md` und `outputs/klima_kommunale_klimaanpassung/answer.md` entstehen.


## Lernkarte: Was du in diesem Notebook siehst

Dieses Notebook ist als praktische Lernstrecke gebaut. Es geht nicht darum, sofort ein produktives System zu bauen, sondern das Architektur-Pattern einmal mit den Händen zu verstehen.

Die vier Schichten sind:

1. **Dokumentgraph**: `Document -> Chunk`. Die Quelle wird in suchbare Textstücke zerlegt.
2. **Ontologiegraph**: `Ontology -> OntologyClass -> Entity/Concept -> ONTOLOGY_RELATION`. Die fachliche Kontextkarte wird als Graph geladen.
3. **Retrievalgraph**: `Question -> Vector Search -> Chunk -> Entity`. Die Frage findet ähnliche Chunks und hängt sie an Ontologie-Kontext.
4. **Antwortsynthese**: `Belege -> LLM -> answer.md`. Das LLM formuliert nur auf Basis der gefundenen Belege.

Hinweis: Das ist GraphRAG mit Neo4j/Cypher und OpenAI Embeddings. Es ist kein GraphQL-Beispiel.


## 0. Vorbereitung

Dieser Abschnitt richtet die Notebook-Umgebung ein: Pfade, Konfiguration, Secrets aus `.env`, Modellparameter und die Szenario-Frage.

**Was der Code macht:** Er lädt Umgebungsvariablen, sucht optional die Neo4j-Aura-Credentials-Datei, setzt Modell- und Suchparameter und definiert die Quelle, die dieses Notebook verarbeiten soll.

**Was du danach sehen solltest:** Eine kurze Konfigurationsübersicht. Wichtig sind `source_patterns`, `output_dir`, `question`, `aura_credentials_loaded` und `neo4j_database`. Daran erkennst du, ob das Notebook mit dem richtigen Szenario arbeitet.

**Szenario-Fokus:** Kommunale Klimaresilienz, Hebelpunkte, Einflussfaktoren, Maßnahmen, Hitze und Starkregen.


In [32]:
from __future__ import annotations

import fnmatch
import json
import os
import re
import textwrap
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Iterable

import pandas as pd
from IPython.display import Markdown, display
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI
from pypdf import PdfReader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
SCENARIO_ID = "klima_kommunale_klimaanpassung"
OUTPUT_DIR = ROOT / "outputs" / SCENARIO_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")


def show_json(title: str, description: str, value: dict) -> None:
    formatted = json.dumps(value, indent=2, ensure_ascii=False)
    display(Markdown(f"### {title}\n\n{description}\n\n```json\n{formatted}\n```"))


def show_table(title: str, description: str, frame: pd.DataFrame) -> None:
    display(Markdown(f"### {title}\n\n{description}"))
    display(frame)


def show_file(title: str, description: str, path: Path) -> None:
    show_json(title, description, {"path": str(path)})


def load_key_value_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        separator = "=" if "=" in line else ":" if ":" in line else None
        if not separator:
            continue
        key, value = line.split(separator, 1)
        values[key.strip()] = value.strip().strip('"')
    return values


def load_neo4j_aura_credentials(root: Path) -> dict[str, str]:
    files = sorted(root.glob("Neo4j-*-Created-*.txt"))
    if not files:
        return {}
    values = load_key_value_file(files[0])
    for key, value in values.items():
        os.environ.setdefault(key, value)
    if "NEO4J_USERNAME" in values:
        os.environ.setdefault("NEO4J_USER", values["NEO4J_USERNAME"])
    return values

AURA_CREDENTIALS = load_neo4j_aura_credentials(ROOT)

OPENAI_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
OPENAI_ANSWER_MODEL = os.getenv("OPENAI_ANSWER_MODEL", "gpt-4.1-mini")
OPENAI_ANSWER_TEMPERATURE = float(os.getenv("OPENAI_ANSWER_TEMPERATURE", "0.2"))
EMBEDDING_DIM = int(os.getenv("OPENAI_EMBEDDING_DIM", "1536"))
TOP_K = int(os.getenv("TOP_K", "8"))
HOP_PENALTY = float(os.getenv("HOP_PENALTY", "0.08"))

DEFAULT_QUESTION = 'Wie entsteht kommunale Klimaresilienz aus Bestandsaufnahme, Einflussfaktoren, Hebelpunkten und Maßnahmen gegen Hitze und Starkregen?'
QUESTION_ENV_KEY = f"{SCENARIO_ID.upper()}_QUESTION"
QUESTION = os.getenv(QUESTION_ENV_KEY, DEFAULT_QUESTION)
SOURCE_PATTERNS = ['48_2024_cc_kommunale_klimaanpassung.pdf']

show_json(
    "Konfiguration",
    "Diese Übersicht zeigt, mit welchem Projektordner, Modell, Suchlimit und welcher Quelle das Notebook gerade arbeitet.",
    {
        "root": str(ROOT),
        "data_dir": str(DATA_DIR),
        "embedding_model": OPENAI_MODEL,
        "answer_model": OPENAI_ANSWER_MODEL,
        "answer_temperature": OPENAI_ANSWER_TEMPERATURE,
        "embedding_dim": EMBEDDING_DIM,
        "top_k": TOP_K,
        "question_env_key": QUESTION_ENV_KEY,
        "question": QUESTION,
        "source_patterns": SOURCE_PATTERNS,
    "output_dir": str(OUTPUT_DIR),
    "aura_credentials_loaded": bool(AURA_CREDENTIALS),
        "neo4j_database": os.getenv("NEO4J_DATABASE", "<default>"),
    },
)

### Konfiguration

Diese Übersicht zeigt, mit welchem Projektordner, Modell, Suchlimit und welcher Quelle das Notebook gerade arbeitet.

```json
{
  "root": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike",
  "data_dir": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/data",
  "embedding_model": "text-embedding-3-small",
  "answer_model": "gpt-5-nano",
  "answer_temperature": 0.2,
  "embedding_dim": 1536,
  "top_k": 8,
  "question_env_key": "KLIMA_KOMMUNALE_KLIMAANPASSUNG_QUESTION",
  "question": "Wie entsteht kommunale Klimaresilienz aus Bestandsaufnahme, Einflussfaktoren, Hebelpunkten und Maßnahmen gegen Hitze und Starkregen?",
  "source_patterns": [
    "48_2024_cc_kommunale_klimaanpassung.pdf"
  ],
  "output_dir": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung",
  "aura_credentials_loaded": true,
  "neo4j_database": "3f153076"
}
```

## 1. Quellen laden und Chunking

Dieser Abschnitt macht aus der Quelldatei eine Liste kleiner Textstücke, die später gesucht und belegt werden können.

**Was der Code macht:** Er liest die konfigurierte PDF-, XML-, Markdown- oder Textquelle aus `data/`, extrahiert Text und zerlegt ihn in überlappende Wort-Chunks.

**Was du danach sehen solltest:** Eine kleine Tabelle mit Anzahl der geladenen Dokumente, Anzahl der Chunks und durchschnittlicher Chunk-Länge. Wenn hier `documents = 1` steht, wurde genau die Szenario-Quelle gefunden.

In [33]:
def xml_text(element: ET.Element) -> str:
    return " ".join(part.strip() for part in element.itertext() if part and part.strip())


def load_xml_text(path: Path) -> str:
    tree = ET.parse(path)
    root = tree.getroot()
    chunks: list[str] = []

    # DocBook XML, wie das BSI-Kompendium, nutzt Namespaces. ElementTree liefert
    # Tags deshalb als {namespace}tag. local_name macht die Struktur lesbarer.
    for element in root.iter():
        local_name = element.tag.rsplit("}", 1)[-1]
        if local_name in {"title", "subtitle", "para", "simpara", "entry"}:
            text = xml_text(element)
            if text:
                chunks.append(text)

    if not chunks:
        chunks.append(xml_text(root))

    return "\n".join(dict.fromkeys(chunks))


def load_source(path: Path) -> str:
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    if path.suffix.lower() == ".xml":
        return load_xml_text(path)
    return path.read_text(encoding="utf-8", errors="ignore")


def chunk_words(text: str, size: int = 850, overlap: int = 140) -> Iterable[str]:
    words = text.split()
    step = max(size - overlap, 1)
    for start in range(0, len(words), step):
        part = words[start : start + size]
        if part:
            yield " ".join(part)


def is_low_signal_chunk(text: str) -> bool:
    normalized = text.lower()
    bibliography_markers = ["doi", "http", "www.", " in: ", " et al.", "s. ", "isbn", "literatur", "verfügbar"]
    marker_count = sum(normalized.count(marker) for marker in bibliography_markers)
    climate_terms = ["klimaanpassung", "kommune", "kommunen", "maßnahme", "maßnahmen", "hebel", "resilienz"]
    has_domain_signal = any(term in normalized for term in climate_terms)
    return marker_count >= 8 and not has_domain_signal

all_source_paths = sorted(
    p for p in DATA_DIR.glob("*")
    if p.suffix.lower() in {".pdf", ".md", ".txt", ".xml"}
)
source_paths = [
    p for p in all_source_paths
    if any(fnmatch.fnmatch(p.name, pattern) for pattern in SOURCE_PATTERNS)
]

if not source_paths:
    raise FileNotFoundError(f"Keine passende Quelle gefunden. Gesucht: {SOURCE_PATTERNS}. Vorhanden: {[p.name for p in all_source_paths]}")

docs = [{"doc_id": p.name, "path": str(p), "text": load_source(p)} for p in source_paths]
raw_chunks = []
for doc in docs:
    for index, content in enumerate(chunk_words(doc["text"])):
        raw_chunks.append({
            "chunk_id": f"{doc['doc_id']}::c{index:03d}",
            "doc_id": doc["doc_id"],
            "content": content.strip(),
        })

chunks = [chunk for chunk in raw_chunks if not is_low_signal_chunk(chunk["content"])]
filtered_chunks = len(raw_chunks) - len(chunks)

source_summary = pd.DataFrame({
    "documents": [len(docs)],
    "raw_chunks": [len(raw_chunks)],
    "usable_chunks": [len(chunks)],
    "filtered_low_signal_chunks": [filtered_chunks],
    "avg_chunk_chars": [round(sum(len(c["content"]) for c in chunks) / max(len(chunks), 1))],
})
show_table(
    "Geladene Quellen und Chunks",
    "Diese Tabelle prüft, ob die gewünschte Quelle gefunden wurde, wie viele Chunks entstanden sind und wie viele offensichtliche Literatur-/Metadaten-Chunks ausgefiltert wurden.",
    source_summary,
)


### Geladene Quellen und Chunks

Diese Tabelle prüft, ob die gewünschte Quelle gefunden wurde, wie viele Chunks entstanden sind und wie viele offensichtliche Literatur-/Metadaten-Chunks ausgefiltert wurden.

,documents,raw_chunks,usable_chunks,filtered_low_signal_chunks,avg_chunk_chars
0,1,29,28,1,6615


## 2. Testgraph leeren

Dieser Abschnitt stellt sicher, dass der folgende Lauf isoliert ist und keine alten Testdaten aus einem vorherigen Szenario in die Ergebnisse hineinspielen.

**Was der Code macht:** Er verbindet sich mit Neo4j und leert die konfigurierte Testdatenbank, sofern `RESET_GRAPH_ON_IMPORT` aktiv ist.

**Was du danach sehen solltest:** Eine kurze Bestätigung mit Datenbankname und `reset_graph_on_import`. Für die isolierten Demo-Läufe sollte dieser Wert `true` sein.

In [34]:
NEO4J_URI = os.environ["NEO4J_URI"]
NEO4J_USER = os.getenv("NEO4J_USER") or os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
RESET_GRAPH_ON_IMPORT = os.getenv("RESET_GRAPH_ON_IMPORT", "true").lower() in {"1", "true", "yes", "ja"}

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def neo4j_session():
    if NEO4J_DATABASE:
        return driver.session(database=NEO4J_DATABASE)
    return driver.session()

with neo4j_session() as session:
    if RESET_GRAPH_ON_IMPORT:
        session.run("MATCH (n) DETACH DELETE n")

show_json(
    "Testgraph zurückgesetzt",
    "Diese Ausgabe zeigt, welche Neo4j-Datenbank verwendet wurde und ob der Graph vor diesem Szenario geleert wurde.",
    {"database": NEO4J_DATABASE or "<default>", "reset_graph_on_import": RESET_GRAPH_ON_IMPORT},
)


### Testgraph zurückgesetzt

Diese Ausgabe zeigt, welche Neo4j-Datenbank verwendet wurde und ob der Graph vor diesem Szenario geleert wurde.

```json
{
  "database": "3f153076",
  "reset_graph_on_import": true
}
```

## 3. Embeddings erzeugen

Dieser Abschnitt übersetzt jeden Text-Chunk in einen numerischen Vektor. Diese Vektoren sind die Grundlage für semantische Suche.

**Was der Code macht:** Er schickt die Chunk-Texte an das konfigurierte OpenAI-Embedding-Modell und hängt den resultierenden Vektor an jeden Chunk.

**Was du danach sehen solltest:** Eine kurze Zusammenfassung mit Anzahl der eingebetteten Chunks und Vektordimension. Beim Default-Modell `text-embedding-3-small` sollte die Dimension `1536` sein.

In [35]:
client = OpenAI()


def embed_batch(texts: list[str], batch_size: int = 96) -> list[list[float]]:
    vectors: list[list[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = client.embeddings.create(
            model=OPENAI_MODEL,
            input=batch,
            encoding_format="float",
        )
        vectors.extend(item.embedding for item in response.data)
    return vectors

vectors = embed_batch([c["content"] for c in chunks])
for chunk, vector in zip(chunks, vectors):
    chunk["embedding"] = vector

show_json(
    "Embeddings erzeugt",
    "Diese Werte zeigen, wie viele Chunks eingebettet wurden und welche Vektordimension das verwendete OpenAI-Modell geliefert hat.",
    {"chunks": len(chunks), "embedding_dimensions": len(chunks[0]["embedding"])},
)


### Embeddings erzeugt

Diese Werte zeigen, wie viele Chunks eingebettet wurden und welche Vektordimension das verwendete OpenAI-Modell geliefert hat.

```json
{
  "chunks": 28,
  "embedding_dimensions": 1536
}
```

## 4. Ontologie laden und verstehen

Dieser Abschnitt macht die Kontextkarte sichtbar, bevor Daten nach Neo4j geschrieben werden.

**Was der Code macht:** Er lädt die passende JSON-Ontologie aus `ontologies/`, prüft Klassen, Entitäten und Relationen, extrahiert einfache Term- und Entity-Treffer aus den Chunks und zeigt dir Tabellen zur Ontologie.

**Was du danach sehen solltest:** Eine kurze Ontologie-Zusammenfassung, eine Klassentabelle, eine Entitätentabelle und eine Relationentabelle. Das ist der Moment, in dem du prüfen kannst: Ergibt diese Kontextkarte fachlich Sinn?


In [36]:
CREATE_SCHEMA_STATEMENTS = [
    "CREATE CONSTRAINT doc_id IF NOT EXISTS FOR (d:Document) REQUIRE d.id IS UNIQUE",
    "CREATE CONSTRAINT chunk_id IF NOT EXISTS FOR (c:Chunk) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT term_name IF NOT EXISTS FOR (t:Term) REQUIRE t.name IS UNIQUE",
    "CREATE CONSTRAINT concept_id IF NOT EXISTS FOR (c:Concept) REQUIRE c.id IS UNIQUE",
    "CREATE CONSTRAINT entity_id IF NOT EXISTS FOR (e:Entity) REQUIRE e.id IS UNIQUE",
    "CREATE CONSTRAINT ontology_id IF NOT EXISTS FOR (o:Ontology) REQUIRE o.id IS UNIQUE",
    "CREATE CONSTRAINT ontology_class_id IF NOT EXISTS FOR (c:OntologyClass) REQUIRE c.id IS UNIQUE",
    """
    CREATE VECTOR INDEX chunk_vec_idx IF NOT EXISTS
    FOR (c:Chunk) ON c.embedding
    OPTIONS { indexConfig: {
      `vector.dimensions`: $dim,
      `vector.similarity_function`: 'cosine',
      `vector.hnsw.m`: 16,
      `vector.hnsw.ef_construction`: 128,
      `vector.quantization.enabled`: false
    }}
    """,
]

ONTOLOGY_FILE_BY_SCENARIO = {
    "klima_kommunale_klimaanpassung": "ontologies/klima_kommunale_klimaanpassung.json",
    "bsi_it_grundschutz": "ontologies/bsi_it_grundschutz.json",
    "sozialbericht_2024": "ontologies/sozialbericht_2024.json",
}

ONTOLOGY_PATH = ROOT / ONTOLOGY_FILE_BY_SCENARIO.get(SCENARIO_ID, f"ontologies/{SCENARIO_ID}.json")


def _read_ontology(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Ontologie-Datei nicht gefunden: {path}")
    payload = json.loads(path.read_text(encoding="utf-8"))
    payload.setdefault("ontology_id", path.stem)
    payload.setdefault("version", "0.0.0")
    payload.setdefault("title", path.stem)
    payload.setdefault("entities", [])
    payload.setdefault("classes", [])
    payload.setdefault("relations", [])
    return payload


def _safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9äöüß]+", "_", value.lower().strip()).strip("_")


def _qualified_node_id(ontology_id: str, local_id: str) -> str:
    return f"{ontology_id}:{_safe_slug(local_id)}"


def load_ontology_model(ontology: dict, scenario_id: str) -> tuple[dict, list[dict], list[dict], list[dict]]:
    ontology_id = str(ontology["ontology_id"])

    class_rows = []
    class_by_id: dict[str, str] = {}
    for item in ontology.get("classes", []):
        raw_id = item.get("id") or item.get("name")
        class_id = _qualified_node_id(ontology_id, str(raw_id))
        class_by_id[str(raw_id)] = class_id
        class_rows.append(
            {
                "id": class_id,
                "ontology_id": ontology_id,
                "name": item.get("name", raw_id),
                "description": item.get("description", ""),
            }
        )

    entity_rows = []
    for item in ontology.get("entities", []):
        raw_id = item.get("id") or item.get("name")
        class_ref = item.get("class_id")
        class_id = class_by_id.get(str(class_ref)) if class_ref is not None else None
        if class_id is None and class_ref is not None:
            class_id = _qualified_node_id(ontology_id, str(class_ref))
            if not any(row["id"] == class_id for row in class_rows):
                class_rows.append(
                    {
                        "id": class_id,
                        "ontology_id": ontology_id,
                        "name": str(class_ref),
                        "description": "",
                    }
                )

        entity_rows.append(
            {
                "id": _qualified_node_id(ontology_id, str(raw_id)),
                "ontology_id": ontology_id,
                "scenario": scenario_id,
                "name": item.get("name", raw_id),
                "description": item.get("description", ""),
                "class_id": class_id,
                "aliases": item.get("aliases", []),
            }
        )

    relation_rows = []
    for item in ontology.get("relations", []):
        source = item.get("source_id")
        target = item.get("target_id")
        if not source or not target:
            continue
        relation_rows.append(
            {
                "id": _qualified_node_id(ontology_id, str(item.get("id") or f"{source}-to-{target}")),
                "ontology_id": ontology_id,
                "label": item.get("label", "bezieht_sich_auf"),
                "source_id": _qualified_node_id(ontology_id, str(source)),
                "target_id": _qualified_node_id(ontology_id, str(target)),
                "description": item.get("description", ""),
                "bidirectional": bool(item.get("bidirectional", False)),
            }
        )

    return ontology, class_rows, entity_rows, relation_rows


def extract_entities_from_text(text: str, entities: list[dict]) -> list[dict]:
    text_lower = text.lower()
    found: list[dict] = []
    seen: set[str] = set()

    for row in entities:
        labels = [row["name"]] + list(row.get("aliases", []))
        matches: list[str] = []
        for label in labels:
            candidate = str(label).lower().strip()
            if candidate and candidate in text_lower:
                matches.append(candidate)

        if matches:
            if row["id"] in seen:
                continue
            seen.add(row["id"])
            found.append(
                {
                    "id": row["id"],
                    "name": row["name"],
                    "class_id": row.get("class_id"),
                    "matched_aliases": sorted(set(matches)),
                }
            )

    return found


def neo4j_session():
    if NEO4J_DATABASE:
        return driver.session(database=NEO4J_DATABASE)
    return driver.session()


def simple_terms(text: str, limit: int = 6) -> list[str]:
    stop = {
        "dass",
        "eine",
        "einer",
        "einen",
        "oder",
        "aber",
        "auch",
        "nicht",
        "this",
        "that",
        "with",
        "from",
        "zur",
        "auf",
        "und",
        "der",
        "die",
        "das",
    }
    tokens = re.findall(r"[A-Za-zÄÖÜäöüß][A-Za-zÄÖÜäöüß\-]{3,}", text.lower())
    terms = [token for token in tokens if token not in stop]
    if not terms:
        return []
    return pd.Series(terms).value_counts().head(limit).index.tolist()


ONTOLOGY = _read_ontology(ONTOLOGY_PATH)
ONTOLOGY, CLASS_ROWS, ENTITY_ROWS, ONTOLOGY_RELATION_ROWS = load_ontology_model(ONTOLOGY, SCENARIO_ID)

term_rows: list[dict] = []
mention_rows: list[dict] = []
matched_entity_rows_by_id: dict[str, dict] = {}
has_entity_rows: list[dict] = []
cooccurrence_rows = set()

for chunk in chunks:
    for term in simple_terms(chunk["content"]):
        term_rows.append({"name": term})
        mention_rows.append({"chunk_id": chunk["chunk_id"], "term": term})

    extracted_entities = extract_entities_from_text(chunk["content"], ENTITY_ROWS)
    for entity in extracted_entities:
        matched_entity_rows_by_id.setdefault(entity["id"], entity)
        has_entity_rows.append(
            {
                "chunk_id": chunk["chunk_id"],
                "entity_id": entity["id"],
                "matched_aliases": entity["matched_aliases"],
            }
        )

    entity_ids = [row["id"] for row in extracted_entities]
    for left_index, left in enumerate(entity_ids):
        for right in entity_ids[left_index + 1 :]:
            pair = tuple(sorted([left, right]))
            cooccurrence_rows.add(pair)

matched_entity_rows = list(matched_entity_rows_by_id.values())
cooccurrence_payload = [{"left_id": left, "right_id": right} for left, right in sorted(cooccurrence_rows)]
class_name_by_local_id = {item.get("id"): item.get("name", item.get("id")) for item in ONTOLOGY.get("classes", [])}
entity_name_by_local_id = {item.get("id"): item.get("name", item.get("id")) for item in ONTOLOGY.get("entities", [])}

ontology_class_table = pd.DataFrame([
    {
        "klasse": item.get("name", item.get("id")),
        "id": item.get("id"),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("classes", [])
])

ontology_entity_table = pd.DataFrame([
    {
        "entität": item.get("name", item.get("id")),
        "klasse": class_name_by_local_id.get(item.get("class_id"), item.get("class_id")),
        "aliases": ", ".join(item.get("aliases", [])[:5]),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("entities", [])
])

ontology_relation_table = pd.DataFrame([
    {
        "von": entity_name_by_local_id.get(item.get("source_id"), item.get("source_id")),
        "relation": item.get("label", "bezieht sich auf"),
        "nach": entity_name_by_local_id.get(item.get("target_id"), item.get("target_id")),
        "beschreibung": item.get("description", ""),
    }
    for item in ONTOLOGY.get("relations", [])
])

show_json(
    "Ontologie geladen",
    "Diese Zahlen beschreiben die Kontextkarte, die gleich vollständig in Neo4j landet. `matched_entities` zeigt, wie viele Ontologie-Entitäten in den aktuellen Chunks über Aliase gefunden wurden.",
    {
        "ontology_path": str(ONTOLOGY_PATH),
        "ontology_id": ONTOLOGY.get("ontology_id"),
        "version": ONTOLOGY.get("version"),
        "title": ONTOLOGY.get("title"),
        "classes": len(ONTOLOGY.get("classes", [])),
        "entities": len(ONTOLOGY.get("entities", [])),
        "relations": len(ONTOLOGY.get("relations", [])),
        "matched_entities_in_chunks": len(matched_entity_rows),
        "chunk_entity_links": len(has_entity_rows),
    },
)

if ONTOLOGY.get("reading_guide"):
    display(Markdown(f"### Leselogik der Ontologie\n\n{ONTOLOGY['reading_guide']}"))

show_table(
    "Klassen: Welche Arten von Knoten gibt es?",
    "Klassen sind keine zusätzlichen Themen, sondern Typen. Sie beantworten: Welche Art von Ding ist eine Entität?",
    ontology_class_table,
)
show_table(
    "Entitäten: Welche fachlichen Knoten liegen im Graph?",
    "Entitäten sind die konkreten fachlichen Begriffe der Kontextkarte. Über Aliase werden sie später mit Chunks verbunden.",
    ontology_entity_table,
)
show_table(
    "Relationen: Wie hängen die Entitäten zusammen?",
    "Relationen sind kuratierte Kanten der Ontologie. Sie sind die eigentliche Kontextkarte, die du später in Neo4j anschauen kannst.",
    ontology_relation_table,
)


### Ontologie geladen

Diese Zahlen beschreiben die Kontextkarte, die gleich vollständig in Neo4j landet. `matched_entities` zeigt, wie viele Ontologie-Entitäten in den aktuellen Chunks über Aliase gefunden wurden.

```json
{
  "ontology_path": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/ontologies/klima_kommunale_klimaanpassung.json",
  "ontology_id": "klima_kommunale_klimaanpassung",
  "version": "2.0.0",
  "title": "Kontextkarte Kommunale Klimaanpassung",
  "classes": 7,
  "entities": 22,
  "relations": 24,
  "matched_entities_in_chunks": 22,
  "chunk_entity_links": 325
}
```

### Leselogik der Ontologie

In der Mitte steht Kommunale Klimaanpassung. Links liegen Risiken und Anlass, rechts Umsetzung und Maßnahmen, oben Wissen und Steuerung, unten Ressourcen und Akteure. Die Relationen erklären, warum ein Knoten für die kommunale Klimaresilienz relevant ist.

### Klassen: Welche Arten von Knoten gibt es?

Klassen sind keine zusätzlichen Themen, sondern Typen. Sie beantworten: Welche Art von Ding ist eine Entität?

,klasse,id,beschreibung
0,Kernkonzept,kernkonzept,Zentrales fachliches Konzept der Kontextkarte.
1,Klimarisiko,risiko,"Klimatische Belastung oder Folge, auf die Komm..."
2,Akteur,akteur,"Organisation oder Gruppe, die Entscheidungen v..."
3,Prozess,prozess,Analytischer oder organisatorischer Ablauf im ...
4,Ressource,ressource,"Voraussetzung für Umsetzung, Steuerung oder Le..."
5,Instrument,instrument,"Planungs-, Steuerungs- oder Umsetzungsinstrument."
6,Hebel,hebel,"Ansatzpunkt, der Veränderung und Transformatio..."


### Entitäten: Welche fachlichen Knoten liegen im Graph?

Entitäten sind die konkreten fachlichen Begriffe der Kontextkarte. Über Aliase werden sie später mit Chunks verbunden.

,entität,klasse,aliases,beschreibung
0,Kommunale Klimaanpassung,Kernkonzept,"kommunale klimaanpassung, klimaanpassung","Übergreifendes Handlungsfeld, in dem Kommunen ..."
1,Kommunale Klimaresilienz,Kernkonzept,"klimaresilienz, resilienz","Fähigkeit einer Kommune, Klimafolgen zu bewält..."
2,Bestandsaufnahme,Prozess,"bestandsaufnahme, status quo, ausgangslage",Analyse des aktuellen Stands kommunaler Klimaa...
3,Einflussfaktoren,Prozess,"einflussfaktoren, hemmnisse, förderfaktoren, b...","Faktoren, die Klimaanpassung in Kommunen erlei..."
4,Hebelpunkte,Hebel,"hebelpunkte, hebel, ansatzpunkte","Wirkungsvolle Ansatzpunkte, mit denen Veränder..."
5,Transformation,Hebel,"transformation, transformativer wandel, wandel","Tiefergehender Wandel von Strukturen, Routinen..."
6,Change Management,Prozess,"change management, veränderungsmanagement","Organisierter Umgang mit Veränderung, Rollen, ..."
7,Hitze,Klimarisiko,"hitze, hitzebelastung, hitzewelle, temperatur","Klimarisiko mit Auswirkungen auf Gesundheit, I..."
8,Starkregen und Überflutung,Klimarisiko,"starkregen, überflutung, hochwasser, niederschlag","Klimarisiko, das Entwässerung, Stadtplanung un..."
9,Kleine und mittlere Kommunen,Akteur,"kleine und mittlere kommunen, kleinere kommune...","Kommunentyp, der im Bericht als besonders rele..."


### Relationen: Wie hängen die Entitäten zusammen?

Relationen sind kuratierte Kanten der Ontologie. Sie sind die eigentliche Kontextkarte, die du später in Neo4j anschauen kannst.

,von,relation,nach,beschreibung
0,Kommunale Klimaanpassung,stärkt,Kommunale Klimaresilienz,Klimaanpassung ist das zentrale Handlungsfeld ...
1,Kommunale Klimaanpassung,adressiert,Hitze,Hitze ist eines der sichtbaren kommunalen Klim...
2,Kommunale Klimaanpassung,adressiert,Starkregen und Überflutung,Starkregen und Überflutung gehören zu den komm...
3,Bestandsaufnahme,identifiziert,Einflussfaktoren,"Die Bestandsaufnahme macht Faktoren sichtbar, ..."
4,Einflussfaktoren,zeigen,Hebelpunkte,Aus Einflussfaktoren lassen sich Hebelpunkte f...
5,Hebelpunkte,ermöglichen,Transformation,Hebelpunkte können Veränderung über Einzelmaßn...
6,Transformation,stärkt,Kommunale Klimaresilienz,Transformationsprozesse erhöhen die Fähigkeit ...
7,Change Management,unterstützt,Transformation,"Change Management hilft, Veränderung organisat..."
8,Kommunalverwaltung,koordiniert,Kommunale Klimaanpassung,Die Verwaltung bündelt Zuständigkeiten und ber...
9,Kommunalpolitik,sichert,Ressourcen und Kapazitäten,Politisches Mandat und Priorität entscheiden m...


## 5. Neo4j Schema und Import

Dieser Abschnitt schreibt Dokumentgraph und Ontologiegraph in Neo4j.

**Was der Code macht:** Er erstellt Constraints und den Vector Index, importiert die vollständige Ontologie, legt Dokumente und Chunks an und verbindet Chunks über `HAS_CONCEPT` mit gefundenen Ontologie-Entitäten.

**Was du danach sehen solltest:** Eine Import-Zusammenfassung. Wichtig sind `ontology_entities`, `matched_entities`, `entity_links`, `chunks` und `reset_graph_on_import`.


In [37]:
with neo4j_session() as session:
    for statement in CREATE_SCHEMA_STATEMENTS:
        session.run(statement, dim=EMBEDDING_DIM)

    session.run(
        """
        MERGE (o:Ontology {id:$id})
        SET o.title = $title,
            o.version = $version,
            o.scenario = $scenario,
            o.domain = $domain,
            o.updated_at = datetime()
        """,
        id=ONTOLOGY["ontology_id"],
        title=ONTOLOGY.get("title", ""),
        version=ONTOLOGY.get("version", ""),
        scenario=SCENARIO_ID,
        domain=ONTOLOGY.get("domain", SCENARIO_ID),
    )

    if CLASS_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MERGE (class:OntologyClass {id:r.id})
            SET class.ontology_id = r.ontology_id,
                class.name = r.name,
                class.description = r.description
            """,
            rows=CLASS_ROWS,
        )
        session.run(
            """
            MATCH (o:Ontology {id:$ontology_id})
            UNWIND $rows AS r
            MATCH (class:OntologyClass {id:r.id})
            MERGE (o)-[:HAS_CLASS]->(class)
            """,
            ontology_id=ONTOLOGY["ontology_id"],
            rows=CLASS_ROWS,
        )

    if ENTITY_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MERGE (entity:Entity:Concept {id:r.id})
            SET entity.ontology_id = r.ontology_id,
                entity.scenario = r.scenario,
                entity.name = r.name,
                entity.description = r.description,
                entity.aliases = r.aliases
            """,
            rows=ENTITY_ROWS,
        )
        class_links = [r for r in ENTITY_ROWS if r.get("class_id")]
        if class_links:
            session.run(
                """
                UNWIND $rows AS r
                MATCH (entity:Entity {id:r.id})
                MATCH (class:OntologyClass {id:r.class_id})
                MERGE (entity)-[:INSTANCE_OF]->(class)
                """,
                rows=class_links,
            )

    if ONTOLOGY_RELATION_ROWS:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (left:Concept {id:r.source_id}), (right:Concept {id:r.target_id})
            MERGE (left)-[rel:ONTOLOGY_RELATION]->(right)
            SET rel.id = r.id,
                rel.label = r.label,
                rel.description = r.description,
                rel.ontology_id = r.ontology_id,
                rel.bidirectional = r.bidirectional,
                rel.scenario = $scenario
            """,
            scenario=SCENARIO_ID,
            rows=ONTOLOGY_RELATION_ROWS,
        )

    session.run(
        "UNWIND $rows AS r MERGE (d:Document {id:r.id}) SET d.name=r.name, d.path=r.path, d.scenario = $scenario",
        scenario=SCENARIO_ID,
        rows=[{"id": d["doc_id"], "name": d["doc_id"], "path": d["path"]} for d in docs],
    )
    session.run(
        """
        UNWIND $rows AS r
        MERGE (c:Chunk {id:r.id})
        SET c.text = r.text,
            c.embedding = r.embedding,
            c.doc_id = r.doc_id,
            c.scenario = r.scenario
        WITH c, r
        MATCH (d:Document {id:r.doc_id})
        MERGE (d)-[:HAS_CHUNK]->(c)
        """,
        rows=[{"id": c["chunk_id"], "doc_id": c["doc_id"], "text": c["content"], "embedding": c["embedding"], "scenario": SCENARIO_ID} for c in chunks],
    )
    if term_rows:
        session.run("UNWIND $rows AS r MERGE (:Term {name:r.name})", rows=term_rows)
        session.run(
            """
            UNWIND $rows AS r
            MATCH (c:Chunk {id:r.chunk_id}), (t:Term {name:r.term})
            MERGE (c)-[:MENTIONS]->(t)
            """,
            rows=mention_rows,
        )
    if has_entity_rows:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (chunk:Chunk {id:r.chunk_id}), (concept:Concept {id:r.entity_id})
            MERGE (chunk)-[rel:HAS_CONCEPT]->(concept)
            SET rel.matched_aliases = r.matched_aliases
            """,
            rows=has_entity_rows,
        )
    if cooccurrence_payload:
        session.run(
            """
            UNWIND $rows AS r
            MATCH (left:Concept {id:r.left_id}), (right:Concept {id:r.right_id})
            MERGE (left)-[:CO_OCCURS_WITH]-(right)
            """,
            rows=cooccurrence_payload,
        )

show_json(
    "Graph importiert",
    "Diese Zusammenfassung zeigt den Dokumentengraphen und den Ontologiegraphen.",
    {
        "documents": len(docs),
        "chunks": len(chunks),
        "terms": len({r["name"] for r in term_rows}),
        "matched_entities": len(matched_entity_rows),
        "entity_links": len(has_entity_rows),
        "cooccurrence_edges": len(cooccurrence_payload),
        "ontology_relations": len(ONTOLOGY_RELATION_ROWS),
        "ontology_id": ONTOLOGY["ontology_id"],
        "ontology_version": ONTOLOGY.get("version", ""),
        "ontology_classes": len(CLASS_ROWS),
        "database": NEO4J_DATABASE or "<default>",
        "reset_graph_on_import": RESET_GRAPH_ON_IMPORT,
    },
)


### Graph importiert

Diese Zusammenfassung zeigt den Dokumentengraphen und den Ontologiegraphen.

```json
{
  "documents": 1,
  "chunks": 28,
  "terms": 61,
  "matched_entities": 22,
  "entity_links": 325,
  "cooccurrence_edges": 228,
  "ontology_relations": 24,
  "ontology_id": "klima_kommunale_klimaanpassung",
  "ontology_version": "2.0.0",
  "ontology_classes": 7,
  "database": "3f153076",
  "reset_graph_on_import": true
}
```

## 6. Graph in Neo4j prüfen

Dieser Abschnitt ist der kleine Reality-Check nach dem Import.

**Was der Code macht:** Er fragt Neo4j direkt ab und zeigt, welche Klassen, Entitäten und Relationen wirklich in der Datenbank liegen. Zusätzlich bekommst du Cypher-Abfragen, die du im Neo4j Browser/Aura Explorer ausführen kannst.

**Was du danach sehen solltest:** Zählwerte und Beispiel-Relationen. Wenn hier Entitäten und Relationen auftauchen, ist die Kontextkarte wirklich im Graph angekommen.


In [38]:

GRAPH_COUNTS_CYPHER = """
MATCH (o:Ontology {id:$ontology_id})
OPTIONAL MATCH (o)-[:HAS_CLASS]->(class:OntologyClass)
WITH o, count(DISTINCT class) AS classes
OPTIONAL MATCH (entity:Entity {ontology_id:$ontology_id})
WITH o, classes, count(DISTINCT entity) AS entities
OPTIONAL MATCH (:Entity {ontology_id:$ontology_id})-[rel:ONTOLOGY_RELATION]->(:Entity {ontology_id:$ontology_id})
RETURN o.id AS ontology_id, o.title AS title, classes, entities, count(rel) AS ontology_relations
"""

GRAPH_RELATIONS_CYPHER = """
MATCH (left:Entity {ontology_id:$ontology_id})-[rel:ONTOLOGY_RELATION]->(right:Entity {ontology_id:$ontology_id})
RETURN left.name AS von, rel.label AS relation, right.name AS nach
ORDER BY von, relation, nach
LIMIT 20
"""

GRAPH_CLASSES_CYPHER = """
MATCH (class:OntologyClass)<-[:INSTANCE_OF]-(entity:Entity {ontology_id:$ontology_id})
RETURN class.name AS klasse, collect(entity.name)[0..12] AS entitaeten
ORDER BY klasse
"""

with neo4j_session() as session:
    graph_counts = session.run(GRAPH_COUNTS_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).single().data()
    graph_relations = session.run(GRAPH_RELATIONS_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).data()
    graph_classes = session.run(GRAPH_CLASSES_CYPHER, ontology_id=ONTOLOGY["ontology_id"]).data()

show_json(
    "Neo4j-Graph geprüft",
    "Diese Zahlen kommen direkt aus Neo4j und bestätigen, dass die Ontologie als eigener Graph importiert wurde.",
    graph_counts,
)
show_table(
    "Beispiel-Relationen aus Neo4j",
    "Diese Kanten sind die kuratierte Kontextkarte. Genau diese Struktur solltest du dir später grafisch in Neo4j ansehen.",
    pd.DataFrame(graph_relations),
)
show_table(
    "Klassen mit Entitäten aus Neo4j",
    "Diese Tabelle zeigt noch einmal den Unterschied: Klassen sind Typen, Entitäten sind die konkreten fachlichen Knoten.",
    pd.DataFrame(graph_classes),
)

display(Markdown(
    "### Cypher zum Selberanschauen in Neo4j (Graph-Ansicht)\n\n"
    "Die folgenden Queries geben Knoten und Beziehungen zurück. Im Neo4j Browser oder Aura Explorer kannst du danach auf die Graph-Ansicht wechseln.\n\n"
    "```cypher\n"
    "// Ontologie-Graph: Klassen, Entitäten und ihre fachlichen Beziehungen.\n"
    "MATCH (o:Ontology)\n"
    "OPTIONAL MATCH (o)-[:HAS_CLASS]->(class:OntologyClass)\n"
    "OPTIONAL MATCH (class)<-[:INSTANCE_OF]-(entity:Entity)\n"
    "OPTIONAL MATCH (entity)-[r:ONTOLOGY_RELATION|CO_OCCURS_WITH]->(neighbor:Entity)\n"
    "RETURN o, class, entity, r, neighbor\n"
    "LIMIT 300;\n"
    "```\n\n"
    "```cypher\n"
    "// Dokumenten-Graph: Quelle, Chunks, gefundene Ontologie-Entitäten und einfache Terms.\n"
    "MATCH (d:Document)-[:HAS_CHUNK]->(chunk:Chunk)\n"
    "OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(concept:Concept)\n"
    "OPTIONAL MATCH (chunk)-[:MENTIONS]->(term:Term)\n"
    "RETURN d, chunk, concept, term\n"
    "LIMIT 250;\n"
    "```\n\n"
    "```cypher\n"
    "// Kombinationsgraph: Dokument-Chunks plus Ontologie-Nachbarschaft.\n"
    "MATCH (d:Document)-[:HAS_CHUNK]->(chunk:Chunk)\n"
    "OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(concept:Concept)\n"
    "OPTIONAL MATCH (concept)-[:INSTANCE_OF]->(class:OntologyClass)\n"
    "OPTIONAL MATCH (concept)-[r:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept)\n"
    "OPTIONAL MATCH (chunk)-[:MENTIONS]->(term:Term)\n"
    "RETURN d AS document, chunk, concept, class, related, r, term\n"
    "LIMIT 260;\n"
    "```"
))


### Neo4j-Graph geprüft

Diese Zahlen kommen direkt aus Neo4j und bestätigen, dass die Ontologie als eigener Graph importiert wurde.

```json
{
  "ontology_id": "klima_kommunale_klimaanpassung",
  "title": "Kontextkarte Kommunale Klimaanpassung",
  "classes": 7,
  "entities": 22,
  "ontology_relations": 24
}
```

### Beispiel-Relationen aus Neo4j

Diese Kanten sind die kuratierte Kontextkarte. Genau diese Struktur solltest du dir später grafisch in Neo4j ansehen.

,von,relation,nach
0,Bestandsaufnahme,identifiziert,Einflussfaktoren
1,Beteiligung und Akzeptanz,erleichtert,Umsetzung
2,Change Management,unterstützt,Transformation
3,Einflussfaktoren,zeigen,Hebelpunkte
4,Gesundheitsschutz,konkretisiert,Maßnahmenplanung
5,Hebelpunkte,ermöglichen,Transformation
6,Hitze,erfordert,Gesundheitsschutz
7,Kleine und mittlere Kommunen,benötigen,Ressourcen und Kapazitäten
8,Klimaanpassungskonzept,verankert,Kommunale Klimaanpassung
9,Klimaanpassungskonzept,übersetzt in,Maßnahmenplanung


### Klassen mit Entitäten aus Neo4j

Diese Tabelle zeigt noch einmal den Unterschied: Klassen sind Typen, Entitäten sind die konkreten fachlichen Knoten.

,klasse,entitaeten
0,Akteur,"[Kleine und mittlere Kommunen, Kommunalverwalt..."
1,Hebel,"[Hebelpunkte, Transformation]"
2,Instrument,"[Klimaanpassungskonzept, Maßnahmenplanung, Mon..."
3,Kernkonzept,"[Kommunale Klimaanpassung, Kommunale Klimaresi..."
4,Klimarisiko,"[Hitze, Starkregen und Überflutung]"
5,Prozess,"[Bestandsaufnahme, Einflussfaktoren, Change Ma..."
6,Ressource,"[Ressourcen und Kapazitäten, Wissen und Daten,..."


### Cypher zum Selberanschauen in Neo4j (Graph-Ansicht)

Die folgenden Queries geben Knoten und Beziehungen zurück. Im Neo4j Browser oder Aura Explorer kannst du danach auf die Graph-Ansicht wechseln.

```cypher
// Ontologie-Graph: Klassen, Entitäten und ihre fachlichen Beziehungen.
MATCH (o:Ontology)
OPTIONAL MATCH (o)-[:HAS_CLASS]->(class:OntologyClass)
OPTIONAL MATCH (class)<-[:INSTANCE_OF]-(entity:Entity)
OPTIONAL MATCH (entity)-[r:ONTOLOGY_RELATION|CO_OCCURS_WITH]->(neighbor:Entity)
RETURN o, class, entity, r, neighbor
LIMIT 300;
```

```cypher
// Dokumenten-Graph: Quelle, Chunks, gefundene Ontologie-Entitäten und einfache Terms.
MATCH (d:Document)-[:HAS_CHUNK]->(chunk:Chunk)
OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(concept:Concept)
OPTIONAL MATCH (chunk)-[:MENTIONS]->(term:Term)
RETURN d, chunk, concept, term
LIMIT 250;
```

```cypher
// Kombinationsgraph: Dokument-Chunks plus Ontologie-Nachbarschaft.
MATCH (d:Document)-[:HAS_CHUNK]->(chunk:Chunk)
OPTIONAL MATCH (chunk)-[:HAS_CONCEPT]->(concept:Concept)
OPTIONAL MATCH (concept)-[:INSTANCE_OF]->(class:OntologyClass)
OPTIONAL MATCH (concept)-[r:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept)
OPTIONAL MATCH (chunk)-[:MENTIONS]->(term:Term)
RETURN d AS document, chunk, concept, class, related, r, term
LIMIT 260;
```

## 7. Vector Search plus Ontologie-Kontext

Dieser Abschnitt stellt die eigentliche Demo-Frage und sucht passende Beleg-Chunks im Graphen.

**Was der Code macht:** Er erzeugt ein Embedding für die Frage, fragt den Neo4j Vector Index ab und sammelt zusätzlich Ontologie-Entitäten, benachbarte Ontologie-Knoten, einfache Terms und Nachbar-Chunks ein.

**Was du danach sehen solltest:** Eine Tabelle der Top-Treffer mit `chunk_id`, `score`, `ontology_entities`, `ontology_neighbors` und `terms`. Gute Treffer sollten fachlich zur Frage passen und sinnvolle Ontologie-Knoten zeigen.


In [39]:
question_vector = embed_batch([QUESTION])[0]

SEARCH_CYPHER = '''
CALL db.index.vector.queryNodes('chunk_vec_idx', $k, $query_vector)
YIELD node, score
WITH node, score
MATCH (node)<-[:HAS_CHUNK]-(doc:Document)
CALL (node) {
  OPTIONAL MATCH (node)-[:MENTIONS]->(term:Term)<-[:MENTIONS]-(neighbor:Chunk)
  RETURN collect(DISTINCT term.name)[0..8] AS terms,
         collect(DISTINCT neighbor.id)[0..8] AS neighbors
}
CALL (node) {
  OPTIONAL MATCH (node)-[:HAS_CONCEPT]->(concept:Concept)
  OPTIONAL MATCH (concept)-[:INSTANCE_OF]->(class:OntologyClass)
  RETURN collect(DISTINCT {id: concept.id, name: concept.name, class_name: class.name})[0..8] AS concepts
}
CALL (node) {
  OPTIONAL MATCH (node)-[:HAS_CONCEPT]->(concept:Concept)-[rel:ONTOLOGY_RELATION|CO_OCCURS_WITH]-(related:Concept)
  OPTIONAL MATCH (related)-[:INSTANCE_OF]->(related_class:OntologyClass)
  RETURN collect(DISTINCT {id: related.id, name: related.name, class_name: related_class.name, relation: coalesce(rel.label, type(rel))})[0..8] AS related_concepts
}
RETURN node.id AS chunk_id,
       doc.id AS doc_id,
       node.text AS text,
       score,
       terms,
       neighbors,
       [concept IN concepts WHERE concept.id IS NOT NULL] AS concepts,
       [concept IN related_concepts WHERE concept.id IS NOT NULL] AS related_concepts
ORDER BY score DESC
LIMIT $k
'''

with neo4j_session() as session:
    rows = session.run(SEARCH_CYPHER, k=TOP_K, query_vector=question_vector).data()


def _normalize_list_value(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [item.strip() for item in value.split(",") if item.strip()]
    if isinstance(value, (list, tuple, set)):
        return [str(item).strip() for item in value if str(item).strip()]
    if isinstance(value, dict):
        items = value.get("items")
        if isinstance(items, (list, tuple, set)):
            return [str(item).strip() for item in items if str(item).strip()]
    return [str(value).strip()] if str(value).strip() else []


def _format_concept_item(item):
    if not isinstance(item, dict):
        return str(item).strip()
    name = str(item.get("name", "")).strip()
    class_name = str(item.get("class_name") or "").strip()
    if name and class_name:
        return f"{name} ({class_name})"
    return name


def _format_related_concept_item(item):
    if not isinstance(item, dict):
        return str(item).strip()
    name = str(item.get("name", "")).strip()
    relation = str(item.get("relation") or "").strip()
    if name and relation:
        return f"{relation}: {name}"
    return name


def _normalize_concepts(value):
    if value is None:
        return []
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return []
        if text.startswith("[") and text.endswith("]"):
            try:
                value = json.loads(text)
            except Exception:
                return [item.strip() for item in text.strip("[]").split(",") if item.strip()]
    if isinstance(value, list):
        out = [_format_concept_item(item) for item in value]
        return [item for item in out if item]
    if isinstance(value, (tuple, set)):
        return _normalize_concepts(list(value))
    if isinstance(value, dict):
        item = _format_concept_item(value)
        return [item] if item else []
    return [str(value)] if str(value).strip() else []


def _normalize_related_concepts(value):
    if value is None:
        return []
    if isinstance(value, list):
        out = [_format_related_concept_item(item) for item in value]
        return [item for item in out if item]
    if isinstance(value, (tuple, set)):
        return _normalize_related_concepts(list(value))
    if isinstance(value, dict):
        item = _format_related_concept_item(value)
        return [item] if item else []
    return _normalize_list_value(value)


search_results = pd.DataFrame([
    {
        "chunk_id": row.get("chunk_id", ""),
        "doc_id": row.get("doc_id", ""),
        "score": round(float(row.get("score", 0.0)), 4),
        "ontology_entities": ", ".join(_normalize_concepts(row.get("concepts", []))),
        "ontology_neighbors": ", ".join(_normalize_related_concepts(row.get("related_concepts", []))),
        "terms": _normalize_list_value(row.get("terms", [])),
    }
    for row in rows
])
show_table(
    "Top-Treffer der Vector Search",
    "Diese Tabelle ist der Kern von GraphRAG: semantische Nähe findet Chunks, der Graph liefert fachlichen Kontext dazu.",
    search_results,
)


### Top-Treffer der Vector Search

Diese Tabelle ist der Kern von GraphRAG: semantische Nähe findet Chunks, der Graph liefert fachlichen Kontext dazu.

,chunk_id,doc_id,score,ontology_entities,ontology_neighbors,terms
0,48_2024_cc_kommunale_klimaanpassung.pdf::c003,48_2024_cc_kommunale_klimaanpassung.pdf,0.8316,"Kommunale Klimaanpassung (Kernkonzept), Kommun...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, kommunale, hebelpunkte, chang..."
1,48_2024_cc_kommunale_klimaanpassung.pdf::c009,48_2024_cc_kommunale_klimaanpassung.pdf,0.8206,"Kommunale Klimaanpassung (Kernkonzept), Kommun...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, hebelpunkte, werden, kommunen]"
2,48_2024_cc_kommunale_klimaanpassung.pdf::c018,48_2024_cc_kommunale_klimaanpassung.pdf,0.8198,"Kommunale Klimaanpassung (Kernkonzept), Kommun...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, climate, change, adaptation, ..."
3,48_2024_cc_kommunale_klimaanpassung.pdf::c002,48_2024_cc_kommunale_klimaanpassung.pdf,0.8176,"Kommunale Klimaanpassung (Kernkonzept), Bestan...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, climate, adaptation, sich]"
4,48_2024_cc_kommunale_klimaanpassung.pdf::c016,48_2024_cc_kommunale_klimaanpassung.pdf,0.8172,"Kommunale Klimaanpassung (Kernkonzept), Bestan...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, kommunale, hebelpunkte, einfl..."
5,48_2024_cc_kommunale_klimaanpassung.pdf::c023,48_2024_cc_kommunale_klimaanpassung.pdf,0.8144,"Kommunale Klimaanpassung (Kernkonzept), Bestan...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[climate, change, adaptation, kommunen, online..."
6,48_2024_cc_kommunale_klimaanpassung.pdf::c000,48_2024_cc_kommunale_klimaanpassung.pdf,0.8082,"Kommunale Klimaanpassung (Kernkonzept), Kommun...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, kommunale, hebelpunkte, clima..."
7,48_2024_cc_kommunale_klimaanpassung.pdf::c004,48_2024_cc_kommunale_klimaanpassung.pdf,0.8077,"Kommunale Klimaanpassung (Kernkonzept), Kommun...","CO_OCCURS_WITH: Transformation, CO_OCCURS_WITH...","[klimaanpassung, werden, ford, otto]"


## 8. Provenienzgraph bauen

Dieser Abschnitt macht aus den Suchergebnissen einen kleinen Provenienzgraphen.

**Was der Code macht:** Er erzeugt Nodes und Edges für Dokumente, Treffer-Chunks, Ontologie-Entitäten, Term-Knoten und Nachbar-Chunks und speichert diese Struktur als `provenance.json`.

**Was du danach sehen solltest:** Eine Bestätigung, wo die JSON-Datei gespeichert wurde. Diese Datei ist die maschinenlesbare Grundlage für die spätere Graphansicht.


In [40]:
nodes = []
edges = set()

for row in rows:
    nodes.append({"id": row["doc_id"], "type": "Document", "label": row["doc_id"], "score": None})
    nodes.append({"id": row["chunk_id"], "type": "Chunk", "label": row["chunk_id"], "score": float(row["score"])})
    edges.add((row["doc_id"], row["chunk_id"], "HAS_CHUNK"))
    for concept in row.get("concepts", []):
        nodes.append({"id": concept["id"], "type": "Concept", "label": concept["name"], "score": None})
        edges.add((row["chunk_id"], concept["id"], "HAS_CONCEPT"))
    for related in row.get("related_concepts", []):
        nodes.append({"id": related["id"], "type": "Concept", "label": related["name"], "score": None})
    for concept in row.get("concepts", []):
        for related in row.get("related_concepts", []):
            if concept["id"] != related["id"]:
                edges.add((concept["id"], related["id"], "ONTOLOGY_RELATION"))
    for term in row["terms"]:
        nodes.append({"id": f"term:{term}", "type": "Term", "label": term, "score": None})
        edges.add((row["chunk_id"], f"term:{term}", "MENTIONS"))
    for neighbor in row["neighbors"]:
        nodes.append({"id": neighbor, "type": "NeighborChunk", "label": neighbor, "score": max(float(row["score"]) - HOP_PENALTY, 0)})
        edges.add((row["chunk_id"], neighbor, "RELATED_1H"))

nodes = list({node["id"]: node for node in nodes}.values())
edges = [{"source": s, "target": t, "type": rel} for s, t, rel in sorted(edges)]

provenance = {
    "query": QUESTION,
    "sources": sorted({row["doc_id"] for row in rows}),
    "ontology": {
        "id": ONTOLOGY.get("ontology_id"),
        "title": ONTOLOGY.get("title"),
        "version": ONTOLOGY.get("version"),
    },
    "chunks": [
        {
            "id": row["chunk_id"],
            "doc_id": row["doc_id"],
            "similarity": float(row["score"]),
            "ontology_entities": row.get("concepts", []),
            "ontology_neighbors": row.get("related_concepts", []),
            "terms": row["terms"],
            "neighbors": row["neighbors"],
            "text_preview": textwrap.shorten(row["text"].replace("\n", " "), width=320, placeholder="..."),
        }
        for row in rows
    ],
    "nodes": nodes,
    "edges": edges,
    "scoring": {"similarity": "cosine", "hop_penalty": HOP_PENALTY},
}

(OUTPUT_DIR / "provenance.json").write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8")
show_file(
    "Provenienz-JSON gespeichert",
    "Diese Datei enthält den maschinenlesbaren Provenienzgraphen mit Dokumenten, Chunks, Ontologie-Entitäten, Terms, Nachbarn und Textvorschauen.",
    OUTPUT_DIR / "provenance.json",
)


### Provenienz-JSON gespeichert

Diese Datei enthält den maschinenlesbaren Provenienzgraphen mit Dokumenten, Chunks, Ontologie-Entitäten, Terms, Nachbarn und Textvorschauen.

```json
{
  "path": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung/provenance.json"
}
```

## 9. SVG direkt anzeigen und als HTML exportieren

Dieser Abschnitt macht den Provenienzgraphen sichtbar, zuerst direkt im Notebook und zusätzlich als eigenständige HTML-Datei.

**Was der Code macht:** Er baut eine statische SVG-Grafik aus Dokumentgraph, Ontologiegraph und Belegpfad, rendert sie im Notebook und speichert dieselbe Ansicht als `outputs/klima_kommunale_klimaanpassung/provenance.html`.

**Was du danach sehen solltest:** Direkt unter der Zelle sollte ein Graph erscheinen. Zusätzlich bekommst du eine Bestätigung, wo die HTML-Datei gespeichert wurde.

In [41]:
from html import escape
from IPython.display import HTML, display


def short_label(label: str, max_chars: int = 30) -> str:
    return label if len(label) <= max_chars else label[: max_chars - 1] + "…"


def distribute(items: list[dict], x: int, y_top: int, y_bottom: int) -> list[dict]:
    if not items:
        return []
    if len(items) == 1:
        ys = [(y_top + y_bottom) / 2]
    else:
        step = (y_bottom - y_top) / (len(items) - 1)
        ys = [y_top + index * step for index in range(len(items))]
    return [{**item, "x": x, "y": round(y, 1), "label_short": short_label(item["label"])} for item, y in zip(items, ys)]


def build_provenance_svg(nodes: list[dict], edges: list[dict]) -> tuple[str, dict[str, int]]:
    groups = {
        "Document": [node for node in nodes if node["type"] == "Document"],
        "Chunk": sorted([node for node in nodes if node["type"] == "Chunk"], key=lambda node: -(node.get("score") or 0)),
        "Concept": sorted([node for node in nodes if node["type"] == "Concept"], key=lambda node: node["label"]),
        "Term": sorted([node for node in nodes if node["type"] == "Term"], key=lambda node: node["label"]),
        "NeighborChunk": sorted([node for node in nodes if node["type"] == "NeighborChunk"], key=lambda node: node["label"]),
    }

    positioned = []
    positioned.extend(distribute(groups["Document"], 110, 365, 365))
    positioned.extend(distribute(groups["Chunk"], 375, 165, 560))
    positioned.extend(distribute(groups["Concept"], 650, 120, 600))
    positioned.extend(distribute(groups["Term"], 950, 95, 620))
    positioned.extend(distribute(groups["NeighborChunk"], 610, 705, 755))
    position_by_id = {node["id"]: node for node in positioned}

    colors = {"Document": "#2563eb", "Chunk": "#16a34a", "Concept": "#7c3aed", "NeighborChunk": "#84cc16", "Term": "#f97316"}
    radii = {"Document": 19, "Chunk": 16, "Concept": 13, "NeighborChunk": 10, "Term": 10}
    edge_colors = {"HAS_CHUNK": "#2563eb", "HAS_CONCEPT": "#7c3aed", "CO_OCCURS_WITH": "#a78bfa", "ONTOLOGY_RELATION": "#7c3aed", "MENTIONS": "#f97316", "RELATED_1H": "#94a3b8"}
    edge_widths = {"HAS_CHUNK": 2.8, "HAS_CONCEPT": 2.0, "CO_OCCURS_WITH": 1.1, "ONTOLOGY_RELATION": 1.6, "MENTIONS": 1.4, "RELATED_1H": 0.9}

    svg_parts = []
    svg_parts.append('<svg viewBox="0 0 1180 820" width="100%" height="820" role="img" aria-label="Graf-Kompass Provenienzgraph">')
    svg_parts.append('<rect x="0" y="0" width="1180" height="820" fill="#f8fafc"/>')

    for x, label in [(110, "Dokument"), (375, "Top-Treffer"), (650, "Ontologie-Entitäten"), (950, "Begriffe"), (610, "1-Hop-Nachbar-Chunks")]:
        y = 105 if label != "1-Hop-Nachbar-Chunks" else 678
        svg_parts.append(f'<text x="{x}" y="{y}" text-anchor="middle" font-size="14" font-weight="700" fill="#334155">{escape(label)}</text>')

    legend = [("Document", "Dokument"), ("Chunk", "Top-Treffer"), ("Concept", "Ontologie-Entität"), ("Term", "Begriff"), ("NeighborChunk", "1-Hop-Chunk")]
    for index, (node_type, label) in enumerate(legend):
        x = 50 + index * 150
        svg_parts.append(f'<circle cx="{x}" cy="38" r="7" fill="{colors[node_type]}" stroke="white" stroke-width="2"/>')
        svg_parts.append(f'<text x="{x + 14}" y="42" font-size="12" fill="#334155">{escape(label)}</text>')

    visible_edges = 0
    for edge in edges:
        source = position_by_id.get(edge["source"])
        target = position_by_id.get(edge["target"])
        if not source or not target:
            continue
        visible_edges += 1
        color = edge_colors.get(edge["type"], "#94a3b8")
        width = edge_widths.get(edge["type"], 1.0)
        opacity = "0.55" if edge["type"] != "RELATED_1H" else "0.28"
        svg_parts.append(
            f'<line x1="{source["x"]:.1f}" y1="{source["y"]:.1f}" x2="{target["x"]:.1f}" y2="{target["y"]:.1f}" '
            f'stroke="{color}" stroke-width="{width}" stroke-opacity="{opacity}"/>'
        )

    for node in positioned:
        color = colors[node["type"]]
        radius = radii[node["type"]]
        title = escape(f"{node['label']} | {node['type']}")
        svg_parts.append(f'<g><title>{title}</title><circle cx="{node["x"]:.1f}" cy="{node["y"]:.1f}" r="{radius}" fill="{color}" stroke="white" stroke-width="2.4"/></g>')
        svg_parts.append(f'<text x="{node["x"]:.1f}" y="{node["y"] + radius + 13:.1f}" text-anchor="middle" font-size="10" fill="#0f172a">{escape(node["label_short"])}</text>')

    svg_parts.append("</svg>")
    summary = {
        "Dokumente": len(groups["Document"]),
        "Top-Treffer": len(groups["Chunk"]),
        "Ontologie-Entitäten": len(groups["Concept"]),
        "Begriffe": len(groups["Term"]),
        "1-Hop-Chunks": len(groups["NeighborChunk"]),
        "Kanten": visible_edges,
    }
    return "\n".join(svg_parts), summary


def build_provenance_html(svg: str, summary: dict[str, int]) -> str:
    metrics = "".join(f'<div class="metric"><strong>{value}</strong><span>{escape(label)}</span></div>' for label, value in summary.items())
    return f'''<!doctype html>
<html lang="de">
<head>
  <meta charset="utf-8" />
  <title>Graf-Kompass Provenienzgraph</title>
  <style>
    body {{ margin: 0; font-family: system-ui, sans-serif; background: #f8fafc; color: #0f172a; }}
    header {{ padding: 22px 28px 8px; border-top: 3px solid #1d4ed8; }}
    h1 {{ margin: 0 0 6px; font-size: 24px; }}
    p {{ margin: 0; color: #475569; max-width: 920px; }}
    .metrics {{ display: flex; gap: 10px; flex-wrap: wrap; padding: 14px 28px 8px; }}
    .metric {{ min-width: 112px; padding: 10px 12px; background: white; border: 1px solid #e2e8f0; border-radius: 7px; box-shadow: 0 1px 2px rgba(15,23,42,.05); }}
    .metric strong {{ display: block; font-size: 20px; line-height: 1; }}
    .metric span {{ display: block; margin-top: 5px; font-size: 12px; color: #64748b; }}
    main {{ padding: 0 20px 28px; }}
    .graph {{ background: white; border: 1px solid #e2e8f0; border-radius: 8px; overflow: hidden; box-shadow: 0 1px 3px rgba(15,23,42,.06); }}
  </style>
</head>
<body>
  <header>
    <h1>Graf-Kompass Provenienzgraph</h1>
    <p>Links steht die Quelle, danach kommen semantische Top-Treffer, Ontologie-Entitäten, einfache Begriffe und 1-Hop-Nachbar-Chunks. Linien zeigen, wodurch Dokumentgraph, Ontologiegraph und Belegpfad zusammenhängen.</p>
  </header>
  <section class="metrics">{metrics}</section>
  <main><section class="graph">{svg}</section></main>
</body>
</html>'''

svg, graph_summary = build_provenance_svg(nodes, edges)
html = build_provenance_html(svg, graph_summary)

notebook_html = html.replace("<!doctype html>", "").replace('<html lang="de">', "").replace("</html>", "")
display(HTML(notebook_html))

(OUTPUT_DIR / "provenance.html").write_text(html, encoding="utf-8")
show_file(
    "Provenienz-HTML gespeichert",
    "Diese Datei enthält dieselbe statische SVG-Graphansicht als eigenständige HTML-Datei für Browser, Demo oder Screenshot.",
    OUTPUT_DIR / "provenance.html",
)

### Provenienz-HTML gespeichert

Diese Datei enthält dieselbe statische SVG-Graphansicht als eigenständige HTML-Datei für Browser, Demo oder Screenshot.

```json
{
  "path": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung/provenance.html"
}
```

## 10. Retrieval-Brief mit Belegen

Dieser Abschnitt erzeugt eine knappe Markdown-Datei mit den wichtigsten Treffern, bevor das LLM daraus eine Antwort formuliert.

**Was der Code macht:** Er nimmt die besten Treffer aus der Vector Search, zeigt daraus lesbare Belege und schreibt sie als `retrieval_brief.md`.

**Was du danach sehen solltest:** Eine Bestätigung, wo `retrieval_brief.md` gespeichert wurde. Diese Datei ist noch keine freie LLM-Antwort, sondern der prüfbare Belegkontext für den nächsten Abschnitt.

In [42]:
SCENARIO_TITLES = {
    "klima_kommunale_klimaanpassung": "GraphRAG-Auswertung: Kommunale Klimaanpassung",
    "bsi_it_grundschutz": "GraphRAG-Auswertung: BSI IT-Grundschutz",
    "sozialbericht_2024": "GraphRAG-Auswertung: Sozialbericht 2024",
}

SCENARIO_INTERPRETATIONS = {
    "klima_kommunale_klimaanpassung": "Die Suche zielt auf kommunale Klimaanpassung, Hebelpunkte, Einflussfaktoren und Maßnahmen für Klimaresilienz. Gute Treffer sollten konkrete Ansatzpunkte für Hitze, Starkregen, Governance, Wissen, Ressourcen und Akzeptanz sichtbar machen.",
    "bsi_it_grundschutz": "Die Suche zielt auf Bausteine, Anforderungen und organisatorische Maßnahmen des IT-Grundschutzes. Gute Treffer sollten zeigen, wie Institutionen Informationssicherheit systematisch strukturieren und welche Anforderungen als konkrete Belege dienen.",
    "sozialbericht_2024": "Die Suche zielt auf gesellschaftliche Themen, Indikatoren und Lebenslagen im Sozialbericht. Gute Treffer sollten zeigen, welche Datenbereiche und Befunde für eine datenbasierte Orientierung relevant sind.",
}

SCENARIO_TAKEAWAYS = {
    "klima_kommunale_klimaanpassung": [
        "Kommunale Klimaanpassung wird vor allem als Zusammenspiel aus Beteiligung, Akzeptanz, Governance, Wissen, Ressourcen und konkreten Maßnahmen sichtbar.",
        "Die Hebelpunkt-Perspektive hilft, nicht nur Einzelmaßnahmen zu sammeln, sondern Voraussetzungen für wirksame Veränderung zu erkennen.",
        "Für eine gute Demo sollte das Retrieval noch stärker zwischen fachlichen Kernpassagen und Literatur-/Randkontext unterscheiden.",
    ],
    "bsi_it_grundschutz": [
        "Der IT-Grundschutz eignet sich gut für eine strukturierte GraphRAG-Demo, weil Bausteine, Anforderungen, Rollen und Maßnahmen bereits als fachliche Ordnung vorliegen.",
        "Nützlich sind Treffer, die konkrete Anforderungen und organisatorische Schritte zeigen, nicht nur allgemeine Begriffsnennungen.",
        "Für die nächste Ausbaustufe sollten XML-Strukturelemente als eigene Graph-Knoten modelliert werden.",
    ],
    "sozialbericht_2024": [
        "Der Sozialbericht eignet sich gut für eine datenorientierte Demo, wenn Themenfelder, Indikatoren und Lebenslagen sauber getrennt werden.",
        "Gute Treffer sollten nicht nur Begriffe nennen, sondern Befunde und gesellschaftliche Zusammenhänge sichtbar machen.",
        "Für die nächste Ausbaustufe sollte das Chunking stärker an Kapiteln, Tabellen und Indikatorabschnitten ausgerichtet werden.",
    ],
}

SCENARIO_DOMAIN_SIGNALS = {
    "klima_kommunale_klimaanpassung": ["klimaanpassung", "kommune", "kommunen", "maßnahme", "maßnahmen", "hebel", "resilienz", "hitze", "starkregen"],
    "bsi_it_grundschutz": ["anforderung", "anforderungen", "baustein", "bausteine", "schutzbedarf", "institution", "informationssicherheit", "grundschutz", "maßnahme"],
    "sozialbericht_2024": ["indikator", "indikatoren", "lebenslage", "lebenslagen", "armut", "bildung", "gesundheit", "einkommen", "bevölkerung", "sozial"],
}

SCENARIO_NEXT_STEPS = {
    "klima_kommunale_klimaanpassung": "Fachlich sollte als Nächstes das Chunking nach Abschnitten und eine kuratiertere Entity-Linking-Logik ergänzt werden, damit Literaturverzeichnis-Treffer weniger Gewicht bekommen und echte Hebelpunkte besser sichtbar werden.",
    "bsi_it_grundschutz": "Fachlich sollte als Nächstes die XML-Struktur stärker genutzt werden: Baustein, Anforderung, Rollen und Maßnahmen sollten eigene Knoten bekommen, statt nur als allgemeine Text-Chunks zu erscheinen.",
    "sozialbericht_2024": "Fachlich sollte als Nächstes nach Kapiteln, Themenfeldern und Indikatoren chunked werden, damit Lebenslagen und Datenbefunde präziser als Belege erscheinen.",
}


def classify_hit(row: dict) -> str:
    text = row["text"].lower()
    bibliography_markers = ["doi", "http", "www.", " in: ", " et al.", "s. ", "isbn", "literatur", "verfügbar"]
    marker_count = sum(text.count(marker) for marker in bibliography_markers)
    domain_signals = SCENARIO_DOMAIN_SIGNALS.get(SCENARIO_ID, [])
    has_domain_signal = any(signal in text for signal in domain_signals)
    if marker_count >= 4 and not has_domain_signal:
        return "eher bibliografischer Treffer"
    if has_domain_signal:
        return "inhaltlich relevanter Treffer"
    return "prüfen"


def clean_snippet(text: str, width: int = 430) -> str:
    text = " ".join(text.replace("\n", " ").split())
    text = text.replace("- ", "")
    return textwrap.shorten(text, width=width, placeholder=" ...")


def evidence_title(row: dict, index: int) -> str:
    terms = [term for term in row.get("terms", []) if len(term) > 3]
    if terms:
        return f"Beleg {index}: {', '.join(terms[:3]).title()}"
    return f"Beleg {index}"


def relevance_sentence(row: dict) -> str:
    terms = [term for term in row.get("terms", []) if len(term) > 3]
    if terms:
        return f"Dieser Treffer ist relevant, weil er im Graph mit {', '.join(terms[:4])} verbunden ist."
    return "Dieser Treffer ist relevant, weil er semantisch nah an der Frage liegt."


def build_answer_markdown(rows: list[dict], question: str, output_dir: Path) -> str:
    relevant_rows = [row for row in rows if classify_hit(row) == "inhaltlich relevanter Treffer"]
    review_rows = [row for row in rows if classify_hit(row) != "inhaltlich relevanter Treffer"]
    lead_rows = relevant_rows[:3] or rows[:3]
    title = SCENARIO_TITLES.get(SCENARIO_ID, "GraphRAG-Auswertung")
    interpretation = SCENARIO_INTERPRETATIONS.get(SCENARIO_ID, "Die Suche zeigt die ähnlichsten Textstellen zur Szenariofrage und verbindet sie mit einfachen Begriffen und Nachbar-Chunks.")
    takeaways = SCENARIO_TAKEAWAYS.get(SCENARIO_ID, [])
    next_step = SCENARIO_NEXT_STEPS.get(SCENARIO_ID, "Als nächstes sollten Chunking und Entity-Linking-Logik fachlich geschärft werden.")

    lines = [
        f"# {title}",
        "",
        "## Frage",
        "",
        question,
        "",
        "## Kurze Einordnung",
        "",
        interpretation,
        "",
        "Diese Datei ist ein lesbarer Antwortentwurf aus den gefundenen Belegen. Sie ist noch keine finale LLM-Antwort, sondern zeigt, was das Retrieval aktuell fachlich hergibt.",
        "",
    ]

    if takeaways:
        lines.extend(["## Was sich aus den Treffern ableiten lässt", ""])
        lines.extend([f"- {item}" for item in takeaways])
        lines.append("")

    lines.extend(["## Beste Belege", ""])
    for index, row in enumerate(lead_rows, 1):
        ontology_entities = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
        ontology_neighbors = _normalize_related_concepts(row.get("related_concepts", [])) if '_normalize_related_concepts' in globals() else []
        lines.extend([
            f"### {evidence_title(row, index)}",
            "",
            relevance_sentence(row),
            "",
            f"> {clean_snippet(row['text'])}",
            "",
            "Technische Spur:",
            f"- Quelle: `{row['doc_id']}`",
            f"- Chunk: `{row['chunk_id']}`",
            f"- Ähnlichkeit: `{row['score']:.3f}`",
            f"- Ontologie-Entitäten: {', '.join(ontology_entities) or 'keine'}",
            f"- Ontologie-Nachbarn: {', '.join(ontology_neighbors) or 'keine'}",
            "",
        ])

    if review_rows:
        lines.extend([
            "## Noch zu prüfen",
            "",
            "Einige Treffer sind semantisch nah, können aber Randkontext, Literaturhinweise oder noch unsauber geschnittene Chunks enthalten. Diese Liste ist vor allem ein Hinweis für die nächste Verbesserung des Chunkings.",
            "",
        ])
        for row in review_rows[:3]:
            lines.append(f"- `{row['chunk_id']}` (`{row['score']:.3f}`): {clean_snippet(row['text'], width=260)}")
        lines.append("")

    lines.extend([
        "## Was der Provenienzgraph zeigt",
        "",
        "Der Graph zeigt, aus welcher Quelle die Treffer stammen und über welche Ontologie-Entitäten, Ontologie-Nachbarn und einfachen Begriffe sie mit Nachbar-Chunks verbunden sind. Er ist damit ein Prüfpfad: Du kannst sehen, ob die Antwort wirklich aus passenden Textstellen kommt oder ob das Retrieval noch Rauschen mitbringt.",
        "",
        "## Nächster sinnvoller Schritt",
        "",
        next_step,
        "",
        "## Dateien",
        "",
        f"- Provenienz-JSON: `{output_dir / 'provenance.json'}`",
        f"- Provenienz-HTML: `{output_dir / 'provenance.html'}`",
    ])
    return "\n".join(lines)

answer_markdown = build_answer_markdown(rows, QUESTION, OUTPUT_DIR)
(OUTPUT_DIR / "retrieval_brief.md").write_text(answer_markdown, encoding="utf-8")
show_file(
    "Retrieval-Brief gespeichert",
    "Diese Markdown-Datei ist der belegte Retrieval-Kontext für die anschließende LLM-Antwort.",
    OUTPUT_DIR / "retrieval_brief.md",
)

### Retrieval-Brief gespeichert

Diese Markdown-Datei ist der belegte Retrieval-Kontext für die anschließende LLM-Antwort.

```json
{
  "path": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung/retrieval_brief.md"
}
```

## 11. LLM-Antwort mit Quellenbelegen

Dieser Abschnitt nutzt das LLM jetzt als antwortgebendes Zusammenfassungsinstrument.

**Was der Code macht:** Er übergibt nur die gefundenen Top-Chunks, Scores und Ontologie-Entitäten an das konfigurierte Antwortmodell. Das Modell soll daraus eine deutsche, quellenbasierte Antwort formulieren und keine unbelegten Zusatzbehauptungen erfinden.

**Was du danach sehen solltest:** `answer.md` mit einer lesbaren Antwort und `answer_context.md` mit dem tatsächlich an das LLM gegebenen Kontext. Damit kannst du später vergleichen, wie sich `TOP_K`, Chunkgröße, Concepts oder Modellwahl auswirken.

In [48]:
def context_row(row: dict, index: int) -> dict:
    concepts = _normalize_concepts(row.get("concepts", [])) if '_normalize_concepts' in globals() else []
    if not concepts:
        raw_concepts = row.get("concepts", [])
        if isinstance(raw_concepts, str):
            concepts = [item.strip() for item in raw_concepts.split(",") if item.strip()]
        elif isinstance(raw_concepts, (list, tuple, set)):
            concepts = [str(item).strip() for item in raw_concepts if str(item).strip()]

    terms = _normalize_list_value(row.get("terms", [])) if '_normalize_list_value' in globals() else []
    if not terms:
        raw_terms = row.get("terms", [])
        if isinstance(raw_terms, str):
            terms = [item.strip() for item in raw_terms.split(",") if item.strip()]
        elif isinstance(raw_terms, (list, tuple, set)):
            terms = [str(item).strip() for item in raw_terms if str(item).strip()]

    return {
        "beleg_nr": index,
        "chunk_id": row.get("chunk_id", ""),
        "quelle": row.get("doc_id", "unbekannt"),
        "aehnlichkeit": round(float(row.get("score", 0.0)), 3),
        "ontology_entities": concepts,
        "ontology_neighbors": _normalize_related_concepts(row.get("related_concepts", [])) if '_normalize_related_concepts' in globals() else [],
        "einfache_terms": terms,
        "text": clean_snippet(row.get("text", ""), width=1400),
    }


llm_context = [context_row(row, index) for index, row in enumerate(rows[: min(TOP_K, 6)], 1)]
context_markdown_parts = []
for item in llm_context:
    context_markdown_parts.append(
        "\n".join(
            [
                f"## Beleg {item['beleg_nr']}: {item['chunk_id']}",
                f"Quelle: `{item['quelle']}`",
                f"Ähnlichkeit: `{item['aehnlichkeit']}`",
                f"Ontologie-Entitäten: {', '.join(item['ontology_entities']) or 'keine'}",
                f"Ontologie-Nachbarn: {', '.join(item['ontology_neighbors']) or 'keine'}",
                "",
                item["text"],
            ]
        )
    )
context_markdown = "\n\n".join(context_markdown_parts)
(OUTPUT_DIR / "answer_context.md").write_text(context_markdown, encoding="utf-8")

system_prompt = """
Du bist ein deutschsprachiger GraphRAG-Assistent für den Graf-Kompass.
Antworte nur auf Basis der gelieferten Belege.
Wenn die Belege für eine Aussage nicht reichen, benenne die Grenze ausdrücklich.
Nutze kurze, klare Abschnitte und verweise bei fachlichen Aussagen auf die Belegnummern.
""".strip()

user_prompt = f"""
Frage:
{QUESTION}

Belege als JSON:
{json.dumps(llm_context, ensure_ascii=False, indent=2)}

Schreibe eine Markdown-Antwort mit diesen Abschnitten:
1. Kurzantwort
2. Wichtige Muster im GraphRAG-Kontext
3. Belegte Aussagen
4. Grenzen der aktuellen Antwort
5. Genutzte Belege
""".strip()

response = None
try:
    response = client.responses.create(
        model=OPENAI_ANSWER_MODEL,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=OPENAI_ANSWER_TEMPERATURE,
    )
except Exception as exc:  # noqa: BLE001
    if "Unsupported parameter: 'temperature'" in str(exc):
        response = client.responses.create(
            model=OPENAI_ANSWER_MODEL,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    else:
        raise

llm_answer = response.output_text.strip()
answer_path = OUTPUT_DIR / "answer.md"
answer_path.write_text(llm_answer, encoding="utf-8")

display(Markdown(llm_answer))
show_json(
    "LLM-Antwort gespeichert",
    "Diese Dateien enthalten die generierte Antwort und den belegten Kontext, der an das Modell übergeben wurde.",
    {"answer": str(answer_path), "answer_context": str(OUTPUT_DIR / "answer_context.md"), "answer_model": OPENAI_ANSWER_MODEL},
)

1) Kurzantwort
Kommunale Klimaresilienz entsteht durch eine systematische Abfolge: Bestandsaufnahme des kommunalen Systems, Identifikation relevanter Einflussfaktoren, Ableitung und Nutzung von Hebelpunkten (Levers) sowie koordiniertem Einsatz von Maßnahmen über mehrere Hebelpunkte – auch gegen Hitze- und Starkregenrisiken. Partizipation, rechtliche Rahmenbedingungen und Governance unterstützen diesen Prozess. (Belege: 1, 2, 3, 4, 5, 6)

2) Wichtige Muster im GraphRAG-Kontext
- Kernbausteine: Bestandsaufnahme, Einflussfaktoren und Hebelpunkte bilden den Kern des kommunalen Klimaanpassungsprozesses; Transformation/Change Management sind zentrale Durchführungsfelder. Hitze as Klimarisiko wird explizit betrachtet. (Belege 1, 3, 5)
- Hebelpunkte als Orientierung: Es gibt eine Systematik von Hebelpunkten, die Orientierung und Priorisierung ermöglichen; mehrfache Hebel wirken kombinatorisch transformational. (Belege 3, 4, 5)
- Mehrpunkten-Ansatz: Transformative Veränderung gelingt, wenn Hebel an möglichst vielen Punkten gleichzeitig wirken; radikale Einzelhebels sind nicht zwingend sinnvoll, auch abseits direkt klimapolitischer Maßnahmen gibt es Relevanz. (Belege 3, 4, 5)
- Governance, Wissen, Ressourcen: Wichtige Hebel sind Werte/Ziele, Governance, Bereitstellung/Austausch von Wissen, Ressourcen und Kapazitäten; Maßnahmen werden als Hebel verstanden. (Belege 4, 5)
- Partizipation und Akteursnetze: Hohe Partizipation von Bevölkerung/Zivilgesellschaft erleichtert Klimaanpassung; Abstimmung mit vielen Akteuren wird als herausfordernd, aber erstrebenswert gesehen. (Beleg 2)
- Rechtlicher Rahmen: Klimaanpassung ist in Gesetzen/Fachplanungen verankert, aber Harmonisierung und klare Rahmenbedingungen fehlen oft; Koordination zwischen Ländern ist wichtig. (Beleg 2)
- Hitze und Starkregen als Treiber: Hitze (Klimarisiko) und Starkregen/Überflutung treten als zentrale Risikofelder auf, die in die Bestandsaufnahme und Hebelpunkte integriert werden. (Belege 1, 3, 6)

3) Belegte Aussagen
- Kernbausteine der kommunalen Klimaanpassung umfassen Bestandsaufnahme, Einflussfaktoren, Hebelpunkte, Transformation/Change Management und Klimarisiken wie Hitze. (Belege 1, 3, 5)
- Eine Hebelpunkten-Systematik dient als Orientierungs- und Priorisierungshilfe, um geeignete Schritte zur Stärkung der Klimaresilienz zu identifizieren. (Belege 3, 4)
- Transformative Wirkung ergibt sich aus dem gleichzeitigen Wirken mehrerer Hebelpunkte; Einzelhebels können wirksam sein, aber systemischer Ansatz ist bevorzugt. (Belege 4, 5)
- Werte/Z Vision, Governance, Wissenstransfer, Ressourcen/Kapazitäten und konkrete Maßnahmen werden als zentrale Hebel betrachtet. (Belege 4, 5)
- Partizipation und Öffentlichkeitsarbeit begünstigen Beschäftigung mit Klimaanpassung; fehlende Akzeptanz wird als Hemmnis gesehen. (Beleg 2)
- Rechtliche Rahmenbedingungen existieren, sind aber länderübergreifend nicht einheitlich koordiniert; klarere Rahmenbedingungen fehlen oft. (Beleg 2)
- Hitze und Starkregen werden explizit als Klimarisiken in den Betrachtungen aufgenommen. (Belege 1, 3, 6)
- Es fehlt in den Belegen ein detaillierter, praxisnaher Maßnahmenkatalog gegen Hitze/Starkregen; die Texte fokussieren eher auf generische Hebel und integrativen Vorgehensbedarf sowie Empirie-Validierung. (Beleg 4)

4) Grenzen der aktuellen Antwort
- Die vorliegenden Belege liefern konzeptionelle Rahmen (Hebelpunkte, Bestandsaufnahme, Einflussfaktoren) und grobe Strukturprinzipien, aber keine detaillierten, praxisnahen Maßnahmenlisten speziell gegen Hitze oder Starkregen. (Belege 1–6)
- Während Maßnahmen als Hebel diskutiert werden, fehlen konkrete Kataloge oder Fallstudien mit konkreten konkreten Handlungsmaßnahmen in den Texten. (Beleg 4)
- Empirische Validierung der identifizierten Hebelpunkte wird empfohlen; konkrete Überprüfungen in Kommunen sind in den Quellen nur allgemein angedeutet. (Beleg 4)

5) Genutzte Belege
- Beleg 1
- Beleg 2
- Beleg 3
- Beleg 4
- Beleg 5
- Beleg 6

Hinweis: Falls du eine konkretere, praxisnahe Liste von Maßnahmen gegen Hitze und Starkregen wünschst, bräuchten wir zusätzliche Quellen mit Fallstudien oder Handlungsleitfäden.

### LLM-Antwort gespeichert

Diese Dateien enthalten die generierte Antwort und den belegten Kontext, der an das Modell übergeben wurde.

```json
{
  "answer": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung/answer.md",
  "answer_context": "/Users/michaelmeierhoff/Code/playground/graphrag/graf-kompass-graphrag-spike/outputs/klima_kommunale_klimaanpassung/answer_context.md",
  "answer_model": "gpt-5-nano"
}
```

## 12. Mini-Experimente: Parameter fühlen

Dieser Abschnitt ist zum Spielen gedacht. Er verändert nichts automatisch, sondern zeigt dir, welche Stellschrauben du bewusst ausprobieren kannst.

**Was der Code macht:** Er zeigt kleine Experimente, die du nacheinander durchführen kannst: Frage ändern, `TOP_K` anpassen, Terms ignorieren, Ontologie-Nachbarn stärker beachten.

**Was du danach sehen solltest:** Eine kleine Experimentierliste. Wähle eine Zeile, ändere den genannten Parameter oben im Notebook oder in `.env`, starte den Kernel neu und führe ab Abschnitt 0 erneut aus.


In [49]:
experiment_steps = pd.DataFrame([
    {
        "experiment": "Frage enger stellen",
        "änderung": "DEFAULT_QUESTION oder die Szenario-Question in `.env` fokussieren",
        "worauf achten": "Werden weniger, aber fachlich klarere Ontologie-Entitäten getroffen?",
    },
    {
        "experiment": "Mehr/weniger Treffer",
        "änderung": "TOP_K zum Beispiel auf 4 oder 12 setzen",
        "worauf achten": "Wird die Antwort präziser oder kommt mehr Rauschen in den Belegkontext?",
    },
    {
        "experiment": "Terms kritisch lesen",
        "änderung": "In Abschnitt 7 `terms` und `ontology_entities` vergleichen",
        "worauf achten": "Welche Kontextsignale sind nur Wörter, welche sind echte Fachknoten?",
    },
    {
        "experiment": "Ontologie im Neo4j Browser öffnen",
        "änderung": "Die Cypher-Abfragen aus Abschnitt 6 ausführen",
        "worauf achten": "Ergibt die Kontextkarte ohne Dokument-Chunks schon fachlich Sinn?",
    },
])
show_table(
    "Experimente für das praktische Gefühl",
    "GraphRAG versteht man am besten, wenn man eine Stellschraube ändert und Retrieval, Graph und Antwort miteinander vergleicht.",
    experiment_steps,
)


### Experimente für das praktische Gefühl

GraphRAG versteht man am besten, wenn man eine Stellschraube ändert und Retrieval, Graph und Antwort miteinander vergleicht.

,experiment,änderung,worauf achten
0,Frage enger stellen,DEFAULT_QUESTION oder die Szenario-Question in...,"Werden weniger, aber fachlich klarere Ontologi..."
1,Mehr/weniger Treffer,TOP_K zum Beispiel auf 4 oder 12 setzen,Wird die Antwort präziser oder kommt mehr Raus...
2,Terms kritisch lesen,In Abschnitt 7 `terms` und `ontology_entities`...,"Welche Kontextsignale sind nur Wörter, welche ..."
3,Ontologie im Neo4j Browser öffnen,Die Cypher-Abfragen aus Abschnitt 6 ausführen,Ergibt die Kontextkarte ohne Dokument-Chunks s...


## 13. Auswertung

Dieser letzte Abschnitt ist deine fachliche Bewertung nach dem Durchlauf.

Prüfe nach dem ersten Lauf:

- Verstehst du den Unterschied zwischen Dokumentgraph, Ontologiegraph, Retrievalgraph und Antwortsynthese?
- Ist der Top-Treffer fachlich plausibel?
- Erzählt der Provenienzgraph einen verständlichen Belegpfad?
- Sind die Ontologie-Entitäten hilfreicher als die einfachen Term-Knoten?
- Kannst du in Neo4j die Kontextkarte unabhängig von den Chunks nachvollziehen?
- Würde sich daraus eine Grafik-Lab-Demo mit Retrieval-, Ontologie- und LLM-Synthese-Pattern bauen lassen?
